# Лабораторная работа 1.

Задача: определить съедобность гриба по его признакам. Целевая колонка `class`: `e` означает съедобный гриб, `p` означает ядовитый.

Цель работы: построить воспроизводимый конвейер с `Pipeline`, `ColumnTransformer` и собственным трансформером.

## 1. Подготовка

In [1]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

root = Path('..')
data_path = root / 'data/raw/mushroom.csv'

## 2. Загрузка данных

In [2]:
if not data_path.exists():
    dataset = fetch_openml(name='mushroom', version=1, as_frame=True)
    dataset.frame.to_csv(data_path, index=False)

df = pd.read_csv(data_path, na_values='?')
print('Размер:', df.shape)
df.head()

Размер: (8124, 23)


,cap-shape,cap-surface,cap-color,bruises%3F,odor,gill-attachment,gill-spacing,gill-size,gill-color,stalk-shape,...,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat,class
0,x,s,n,t,p,f,c,n,k,e,...,w,w,p,w,o,p,k,s,u,p
1,x,s,y,t,a,f,c,b,k,e,...,w,w,p,w,o,p,n,n,g,e
2,b,s,w,t,l,f,c,b,n,e,...,w,w,p,w,o,p,n,n,m,e
3,x,y,w,t,p,f,c,n,n,e,...,w,w,p,w,o,p,k,s,u,p
4,x,s,g,f,n,f,w,b,k,t,...,w,w,p,w,o,e,n,a,g,e


In [3]:
df.info()
print('Пропуски:')
print(df.isna().sum())
print('Количество объектов каждого класса:')
print(df['class'].value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 8124 entries, 0 to 8123
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   cap-shape                 8124 non-null   str  
 1   cap-surface               8124 non-null   str  
 2   cap-color                 8124 non-null   str  
 3   bruises%3F                8124 non-null   str  
 4   odor                      8124 non-null   str  
 5   gill-attachment           8124 non-null   str  
 6   gill-spacing              8124 non-null   str  
 7   gill-size                 8124 non-null   str  
 8   gill-color                8124 non-null   str  
 9   stalk-shape               8124 non-null   str  
 10  stalk-root                5644 non-null   str  
 11  stalk-surface-above-ring  8124 non-null   str  
 12  stalk-surface-below-ring  8124 non-null   str  
 13  stalk-color-above-ring    8124 non-null   str  
 14  stalk-color-below-ring    8124 non-null   str  
 15

## 3. Разбиение данных

Выделяем 80% строк для обучения и 20% для проверки. Ядовитый гриб обозначим единицей, съедобный нулём.

In [4]:
X = df.drop(columns='class')
y = df['class'].map({'e': 0, 'p': 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Обучающая выборка:', X_train.shape)
print('Тестовая выборка:', X_test.shape)

Обучающая выборка: (6499, 22)
Тестовая выборка: (1625, 22)


## 4. Трансформер

Считаем редкими значения с частотой меньше 10% в обучающей выборке. Каждое значение `odor` в этом датасете состоит из одного символа, поэтому количество равно 0 или 1. Трансформер добавляет числовой признак `odor_rare_count`, сохраняя исходный `odor`. Частоты определяются в `fit`, без использования тестовых данных.

In [5]:
class RareOdorCounter(TransformerMixin, BaseEstimator):
    def __init__(self, threshold=0.1):
        self.threshold = threshold

    def fit(self, X, y=None):
        frequencies = X['odor'].value_counts(normalize=True)
        self.rare_symbols_ = frequencies[
            frequencies < self.threshold
        ].index
        return self

    def transform(self, X):
        X = X.copy()
        X['odor_rare_count'] = X['odor'].isin(
            self.rare_symbols_
        ).astype(int)
        return X

## 5. Пайплайн обработки и модель

Для исходных категориальных признаков используем заполнение пропусков модой и `OneHotEncoder(handle_unknown='ignore')`.

Для добавленного числового признака используем заполнение медианой и `StandardScaler`.

В конце пайплайна находится логистическая регрессия.

In [6]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, ['odor_rare_count']),
    ('categorical', categorical_pipeline, X.columns.tolist()),
])

model = Pipeline([
    ('rare_odor', RareOdorCounter()),
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42)),
])

## 6. Обучение и оценка

Accuracy показывает долю правильных ответов, ROC-AUC оценивает разделение классов по вероятности ядовитости. Обе метрики считаются на отложенной тестовой выборке.

In [7]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print(f'Accuracy: {accuracy:.4f}')
print(f'ROC-AUC: {roc_auc:.4f}')
print('Редкие символы odor:',
      model.named_steps['rare_odor'].rare_symbols_.tolist())

Accuracy: 0.9988
ROC-AUC: 1.0000
Редкие символы odor: ['y', 's', 'l', 'a', 'p', 'c', 'm']


## 7. Сохранение модели

Сохраняем весь пайплайн в `models/mushroom.joblib`.

In [9]:
joblib.dump(model, root / 'models/mushroom.joblib')

['../models/mushroom.joblib']